In [ ]:
import os
import ctypes
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Try loading libtpu for TPU
try:
    ctypes.CDLL("libtpu.so", mode=ctypes.RTLD_GLOBAL)
    print("[TPU] libtpu.so loaded")
except Exception as e:
    print(f"[TPU] libtpu.so not found: {e}")

import jax
import jax.numpy as jnp
from jax import random

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
N_DEVICES = jax.device_count()
print(f"Device count: {N_DEVICES}")

# Keep TF for data loading only
import tensorflow as tf
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

BATCH_SIZE_PER_DEVICE = 16
EFFECTIVE_BATCH = BATCH_SIZE_PER_DEVICE * N_DEVICES
print(f"Batch: {BATCH_SIZE_PER_DEVICE}/device x {N_DEVICES} devices = {EFFECTIVE_BATCH} effective")
if N_DEVICES > 1:
    print("TPU DETECTED - training on TPU!")
else:
    print("No TPU. Training on CPU/GPU.")


In [ ]:
import os

DATA_DIR = None
base = "/kaggle/input/datasets"
for user in os.listdir(base):
    for dataset in os.listdir(os.path.join(base, user)):
        candidate = os.path.join(base, user, dataset, "dataset")
        if os.path.isdir(os.path.join(candidate, "train")):
            DATA_DIR = candidate
            break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find dataset under /kaggle/input/datasets.")

print(f"DATA_DIR = {DATA_DIR}")


In [ ]:
import os
import tensorflow as tf

try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/egyptian-new-currency-2023/dataset"
    if not os.path.exists(DATA_DIR):
        DATA_DIR = "./egyptian-new-currency-2023/dataset"
    print(f"DATA_DIR not set. Using fallback: {DATA_DIR}")

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.gif')
data_dir = os.path.join(DATA_DIR, 'train')

print("Verifying image integrity...")
removed_count = 0

for root, dirs, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.lower().endswith(valid_extensions):
            try:
                os.remove(file_path)
                removed_count += 1
                continue
            except: pass
        try:
            img_bytes = tf.io.read_file(file_path)
            tf.io.decode_image(img_bytes)
        except Exception:
            print(f"Removing corrupted file: {file_path}")
            try:
                os.remove(file_path)
                removed_count += 1
            except: pass

print(f"Cleanup complete. Removed {removed_count} problematic files.")

## RUN 4: REVERT TO RUN 2 CONFIG + MORE AUGMENTATION
### Fix from Run 3 Catastrophe:
- REVERT to Run 2 architecture (no L2!)
- Keep LR at 1e-4 (NOT 5e-5)
- Add more augmentation only
- Same dropout rates as Run 2
- _Run 4 already completed (69.01%) - placeholder cells below_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import datetime
import os

IMG_HEIGHT = 128
IMG_WIDTH = 128

train_dir = os.path.join(DATA_DIR, "train")

# Load with tf.data (batching, prefetch on CPU; model runs on TPU)
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=EFFECTIVE_BATCH)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=EFFECTIVE_BATCH)

class_names = train_ds_raw.class_names
NUM_CLASSES = len(class_names)
print(f"Classes ({NUM_CLASSES}): {class_names}")

# Augmentation pipeline on CPU via tf.data.map
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

def preprocess_train(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = aug(x, training=True)
    return x, tf.cast(y, tf.int32)

def preprocess_val(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    return x, tf.cast(y, tf.int32)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds_raw.map(preprocess_val, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Ensure static batch size for XLA/TPU
train_ds = train_ds.unbatch().batch(EFFECTIVE_BATCH, drop_remainder=True)
val_ds = val_ds.unbatch().batch(EFFECTIVE_BATCH, drop_remainder=True)

# Steps per epoch: use raw dataset length (drop_remainder makes len() unknown)
steps_per_epoch = len(train_ds_raw)
print(f"Train batches: ~{steps_per_epoch}, Val batches: ~{len(val_ds_raw)}")

# Load held-out valid split for final test evaluation
valid_dir = os.path.join(DATA_DIR, "valid")
if os.path.isdir(valid_dir):
    test_ds = tf.keras.utils.image_dataset_from_directory(
        valid_dir, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=EFFECTIVE_BATCH)
    test_ds = test_ds.map(preprocess_val, num_parallel_calls=AUTOTUNE)
    test_ds = test_ds.unbatch().batch(EFFECTIVE_BATCH, drop_remainder=True).prefetch(AUTOTUNE)
    print(f"Loaded held-out valid split")
else:
    test_ds = val_ds
    print("No valid/ split. Using val_ds for final evaluation.")


In [ ]:
print("Run 4 already completed previously. Skipping to Run 5.")


In [ ]:
all_run_metrics = []
best_overall_acc = 0.6901
best_overall_run = 5
best_history = None
print("Run 4 already completed previously (best: 69.01%). Proceeding to Run 5.")


In [ ]:
print("Run 4 visualizations were saved in previous session. Skipping.")


In [ ]:
print("Run 4 accuracy/loss curves were saved in previous session. Skipping.")


In [ ]:
print("Run 4 classification report was saved in previous session. Skipping.")


In [ ]:
print("Run 4 model already saved from previous session. Skipping.")


## RUN 5: VGG-STYLE CNN IN JAX/FLAX TARGETING 93%
### Config:
- VGG-style Flax nn.Module: 4 conv blocks (64->128->256->512)
- GlobalAveragePooling instead of Flatten (reduces overfitting)
- Input: 128x128
- Up to 15 restarts x 50 epochs (stops early if 93% hit)
- Adam + CosineDecay (1e-3 -> 1e-6) + gradient clipping
- Label smoothing 0.1
- EarlyStopping (patience=10)
- jax.pmap across all TPU devices
- Augmentation on CPU via tf.data pipeline
- Ensemble top 3 models


In [ ]:
import datetime
RUN5_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN5_DIR = f"./visualizations/run5_{RUN5_TIMESTAMP}"
os.makedirs(RUN5_DIR, exist_ok=True)
print(f"Run 5 outputs -> {RUN5_DIR}")


In [ ]:
import functools
from flax import linen as nn
from flax.training import train_state
import optax

class VGG(nn.Module):
    '''VGG-style CNN: 4 conv blocks + GAP head.'''
    num_classes: int

    @nn.compact
    def __call__(self, x, train: bool = True):
        # Block 1: 64 x 2
        for _ in range(2):
            x = nn.Conv(64, (3, 3), padding='SAME')(x)
            x = nn.BatchNorm(use_running_average=not train)(x)
            x = nn.relu(x)
        x = nn.max_pool(x, (2, 2), (2, 2))
        x = nn.Dropout(0.2, deterministic=not train)(x)

        # Block 2: 128 x 2
        for _ in range(2):
            x = nn.Conv(128, (3, 3), padding='SAME')(x)
            x = nn.BatchNorm(use_running_average=not train)(x)
            x = nn.relu(x)
        x = nn.max_pool(x, (2, 2), (2, 2))
        x = nn.Dropout(0.25, deterministic=not train)(x)

        # Block 3: 256 x 3
        for _ in range(3):
            x = nn.Conv(256, (3, 3), padding='SAME')(x)
            x = nn.BatchNorm(use_running_average=not train)(x)
            x = nn.relu(x)
        x = nn.max_pool(x, (2, 2), (2, 2))
        x = nn.Dropout(0.3, deterministic=not train)(x)

        # Block 4: 512 x 3
        for _ in range(3):
            x = nn.Conv(512, (3, 3), padding='SAME')(x)
            x = nn.BatchNorm(use_running_average=not train)(x)
            x = nn.relu(x)
        x = nn.max_pool(x, (2, 2), (2, 2))
        x = nn.Dropout(0.4, deterministic=not train)(x)

        # Head
        x = x.mean(axis=(1, 2))  # GlobalAveragePooling2D
        x = nn.Dense(512)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Dropout(0.5, deterministic=not train)(x)
        x = nn.Dense(256)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Dropout(0.3, deterministic=not train)(x)
        x = nn.Dense(num_classes)(x)
        return x

# Optimizer with CosineDecay + gradient clipping
EPOCHS_5 = 50
total_steps = steps_per_epoch * EPOCHS_5
lr_schedule = optax.cosine_decay_schedule(
    init_value=1e-3, decay_steps=total_steps, alpha=1e-6 / 1e-3
)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_schedule),
)

# TrainState holding params + batch_stats
class TrainState(train_state.TrainState):
    batch_stats: dict = None

def create_train_state(rng, num_classes):
    model = VGG(num_classes=num_classes)
    variables = model.init(rng, jnp.ones((1, IMG_HEIGHT, IMG_WIDTH, 3)), train=True)
    state = TrainState.create(
        apply_fn=model.apply,
        params=variables.get('params', {}),
        tx=optimizer,
        batch_stats=variables.get('batch_stats', {}),
    )
    return state

# Test instantiation
rng = jax.random.PRNGKey(0)
_demo_state = create_train_state(rng, NUM_CLASSES)
param_count = sum(x.size for x in jax.tree_util.tree_leaves(_demo_state.params))
print(f"VGG model created. Parameters: {param_count:,}")
print(f"LR schedule: {lr_schedule(0):.6f} -> {lr_schedule(total_steps):.6f}")


In [ ]:
import functools
import pickle
import numpy as np

# ── Batch preparation for pmap ──
def prepare_batch(batch_x, batch_y):
    '''Reshape flat batch to (N_DEVICES, bs_per_device, ...) for pmap.'''
    def _reshape(arr):
        return arr.reshape((N_DEVICES, -1) + arr.shape[1:])
    return _reshape(batch_x), _reshape(batch_y)

def unreplicate_state(state):
    '''Get first-device copy of replicated state.'''
    return jax.device_get(jax.tree_util.tree_map(lambda x: x[0], state))

def replicate_state(state):
    return jax.device_put_replicated(state, jax.local_devices())

# ── pmap train step ──
@functools.partial(jax.pmap, axis_name='devices')
def train_step(state, batch, rng):
    x, y = batch

    def loss_fn(params):
        variables = {'params': params, 'batch_stats': state.batch_stats}
        logits, updates = state.apply_fn(
            variables, x, train=True, mutable=['batch_stats'],
            rngs={'dropout': rng})
        y_onehot = jax.nn.one_hot(y, NUM_CLASSES)
        loss = optax.softmax_cross_entropy(logits, y_onehot, label_smoothing=0.1).mean()
        return loss, (logits, updates['batch_stats'])

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (logits, new_batch_stats)), grads = grad_fn(state.params)

    loss = jax.lax.pmean(loss, axis_name='devices')
    grads = jax.lax.pmean(grads, axis_name='devices')
    acc = jnp.mean(jnp.argmax(logits, -1) == y)
    acc = jax.lax.pmean(acc, axis_name='devices')

    state = state.apply_gradients(grads=grads, batch_stats=new_batch_stats)
    return state, loss, acc

# ── pmap eval step ──
@functools.partial(jax.pmap, axis_name='devices')
def eval_step(state, batch):
    x, y = batch
    variables = {'params': state.params, 'batch_stats': state.batch_stats}
    logits = state.apply_fn(variables, x, train=False, mutable=False)
    return logits, y

@functools.partial(jax.pmap, axis_name='devices')
def eval_step_softmax(state, batch):
    x, y = batch
    variables = {'params': state.params, 'batch_stats': state.batch_stats}
    logits = state.apply_fn(variables, x, train=False, mutable=False)
    probs = jax.nn.softmax(logits, axis=-1)
    return probs, y

def evaluate(state, dataset):
    '''Gather predictions/labels from all devices across entire dataset.'''
    all_logits, all_labels = [], []
    total_samples = 0
    for batch_x, batch_y in dataset.as_numpy_iterator():
        if batch_x.shape[0] < EFFECTIVE_BATCH:
            continue
        total_samples += batch_x.shape[0]
        batch_x, batch_y = prepare_batch(batch_x, batch_y)
        logits, labels = eval_step(state, (batch_x, batch_y))
        logits = jax.device_get(logits).reshape(-1, logits.shape[-1])
        labels = jax.device_get(labels).reshape(-1)
        all_logits.append(logits)
        all_labels.append(labels)
    if not all_logits:
        return np.array([]), np.array([]), np.array([]), 0.0
    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = np.argmax(all_logits, -1)
    acc = (preds == all_labels).mean()
    return preds, all_labels, all_logits, acc

def evaluate_softmax(state, dataset):
    '''Same as evaluate but returns softmax probs (for ensemble).'''
    all_probs, all_labels = [], []
    for batch_x, batch_y in dataset.as_numpy_iterator():
        if batch_x.shape[0] < EFFECTIVE_BATCH:
            continue
        batch_x, batch_y = prepare_batch(batch_x, batch_y)
        probs, labels = eval_step_softmax(state, (batch_x, batch_y))
        probs = jax.device_get(probs).reshape(-1, probs.shape[-1])
        labels = jax.device_get(labels).reshape(-1)
        all_probs.append(probs)
        all_labels.append(labels)
    if not all_probs:
        return np.array([]), np.array([]), 0.0
    return np.concatenate(all_probs, axis=0), np.concatenate(all_labels, axis=0)

# ── Early Stopping ──
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.best_acc = 0.0
        self.counter = 0
        self.should_stop = False

    def update(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

# ── Checkpointing ──
def save_ckpt(state_replicated, path):
    ckpt = jax.device_get(unreplicate_state(state_replicated))
    with open(path, 'wb') as f:
        pickle.dump({'params': ckpt.params, 'batch_stats': ckpt.batch_stats}, f)

def load_ckpt(state_replicated, path):
    with open(path, 'rb') as f:
        ckpt = pickle.load(f)
    state_unrepl = unreplicate_state(state_replicated)
    state_loaded = state_unrepl.replace(
        params=ckpt['params'],
        batch_stats=ckpt['batch_stats'],
    )
    return replicate_state(state_loaded)

# ── Main training loop ──
MAX_RUNS_5 = 15
best5_acc = 0.0
best5_state = None
best5_run = 0
all5_metrics = []
all5_states = []
run_num = 0

print(f"{'='*60}")
print(f"RUN 5: JAX VGG | UP TO {MAX_RUNS_5} RESTARTS x {EPOCHS_5} EPOCHS")
print(f"Adam + CosineDecay (1e-3 -> 1e-6) | LabelSmoothing=0.1")
print(f"Devices: {N_DEVICES} | Batch: {BATCH_SIZE_PER_DEVICE}/dev = {EFFECTIVE_BATCH} eff")
print(f"Target: 93% | Stop early if met")
print(f"{'='*60}\n")

while best5_acc < 0.93 and run_num < MAX_RUNS_5:
    run_num += 1
    print(f"\n{'='*60}")
    print(f"RUN {run_num}/{MAX_RUNS_5}")
    print(f"{'='*60}")

    rng = jax.random.PRNGKey(run_num * 123)
    state = create_train_state(rng, NUM_CLASSES)
    state = replicate_state(state)
    es = EarlyStopping(patience=10)

    for epoch in range(1, EPOCHS_5 + 1):
        epoch_loss, epoch_acc, n_batches = 0.0, 0.0, 0
        for batch_x, batch_y in train_ds.as_numpy_iterator():
            if batch_x.shape[0] < EFFECTIVE_BATCH:
                continue
            batch_x, batch_y = prepare_batch(batch_x, batch_y)
            rng, step_rng = jax.random.split(rng)
            step_rng = jax.random.split(step_rng, N_DEVICES)
            state, loss, acc = train_step(state, (batch_x, batch_y), step_rng)
            epoch_loss += float(loss[0])
            epoch_acc += float(acc[0])
            n_batches += 1

        epoch_loss /= max(n_batches, 1)
        epoch_acc /= max(n_batches, 1)

        _, _, _, val_acc = evaluate(state, val_ds)

        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS_5:
            print(f"  Epoch {epoch:3d}: train_acc={epoch_acc:.4f} loss={epoch_loss:.4f} val_acc={val_acc:.4f}")

        is_best = es.update(val_acc)
        if is_best:
            ckpt_path = f"{RUN5_DIR}/run{run_num}_best.pkl"
            save_ckpt(state, ckpt_path)
        if es.should_stop:
            print(f"  Early stopping at epoch {epoch} (best={es.best_acc:.4f})")
            break

    # Load best checkpoint
    ckpt_path = f"{RUN5_DIR}/run{run_num}_best.pkl"
    if os.path.exists(ckpt_path):
        state = load_ckpt(state, ckpt_path)

    _, _, _, final_val = evaluate(state, val_ds)
    metrics = {"run": run_num, "best": es.best_acc, "final_val": final_val}
    all5_metrics.append(metrics)
    all5_states.append(state)

    marker = ""
    if es.best_acc > best5_acc:
        best5_acc = es.best_acc
        best5_run = run_num
        best5_state = state
        marker = "  <<< BEST"
    if es.best_acc >= 0.93:
        marker += "  *** TARGET MET! ***"

    print(f"  BEST={es.best_acc:.4f}  FINAL={final_val:.4f}{marker}")

    if run_num % 2 == 0 or best5_acc >= 0.93:
        print(f"\n>>> PROGRESS (after {run_num} runs) <<<")
        for m in all5_metrics:
            print(f"  Run {m['run']}: Best={m['best']:.4f}  Final={m['final_val']:.4f}")
        print(f"  >> Overall best: {best5_acc:.4f} | Target: 0.93 | Gap: {max(0, 0.93 - best5_acc):.4f}")

    if best5_acc >= 0.93:
        break

print(f"\n{'='*60}")
if best5_acc >= 0.93:
    print(f"*** TARGET 93% MET! Best: {best5_acc:.4f} (Run {best5_run}) ***")
else:
    print(f"Stopped after {run_num} runs. Best: {best5_acc:.4f} | Gap: {0.93 - best5_acc:.4f}")
print(f"{'='*60}")


In [ ]:
import numpy as np

run_nums = [m['run'] for m in all5_metrics]
best_vals = [m['best'] for m in all5_metrics]
final_vals = [m['final_val'] for m in all5_metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(run_nums, best_vals, 'b-o', label='Best Val Acc', linewidth=2, markersize=8)
axes[0].plot(run_nums, final_vals, 'r--s', label='Final Val Acc', linewidth=2, markersize=8)
axes[0].axhline(y=0.93, color='g', linestyle=':', linewidth=2, label='Target: 93%')
axes[0].set_xlabel('Run Number', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title(f'Run 5: JAX VGG ({run_num} Runs)', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].bar(run_nums, best_vals, color='steelblue', alpha=0.7)
axes[1].axhline(y=0.93, color='r', linestyle='--', linewidth=2, label='Target: 93%')
axes[1].axhline(y=best5_acc, color='darkorange', linestyle='-', linewidth=2,
                label=f'Best: {best5_acc:.4f}')
axes[1].set_xlabel('Run Number', fontsize=12)
axes[1].set_ylabel('Best Validation Accuracy', fontsize=12)
axes[1].set_title('Best Accuracy per Run', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Best: {best5_acc:.4f} (Run {best5_run}) | Target: 0.93 | Gap: {max(0, 0.93 - best5_acc):.4f}")


In [ ]:
print("Per-epoch accuracy curves not tracked in JAX version. See comparison plots above.")


In [ ]:
print("\n--- RUN 5 Classification Report (Best Model on Test Set) ---")

best_state_eval = replicate_state(unreplicate_state(best5_state))
preds, labels, logits, test_acc = evaluate(best_state_eval, test_ds)
if len(preds) == 0:
    preds, labels, logits, test_acc = evaluate(best_state_eval, val_ds)
    print("(Falling back to val_ds for evaluation)")

print(f"Test Accuracy (best single model): {test_acc:.4f}")
print(classification_report(labels, preds, target_names=class_names))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Run 5 Confusion Matrix (Best Single Model)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
print("\n--- RUN 5 Ensemble Prediction (Top 3 Models on Test Set) ---")

sorted_runs = sorted(zip(all5_metrics, all5_states), key=lambda x: -x[0]['best'])
top3 = sorted_runs[:min(3, len(sorted_runs))]

all_probs = []
for i, (metrics, state) in enumerate(top3):
    st_eval = replicate_state(unreplicate_state(state))
    probs, _ = evaluate_softmax(st_eval, test_ds if len(test_ds) > 0 else val_ds)
    all_probs.append(probs)
    print(f"  Model {i+1} (val_acc={metrics['best']:.4f}): included")

avg_probs = np.mean(all_probs, axis=0)
ensemble_preds = np.argmax(avg_probs, -1)

# Recompute labels if needed
if len(labels) != len(ensemble_preds):
    _, labels, _, _ = evaluate(best_state_eval, test_ds if len(test_ds) > 0 else val_ds)

ensemble_acc = (ensemble_preds == labels[:len(ensemble_preds)]).mean()

print(f"\nEnsemble (top {len(top3)}) Test Accuracy: {ensemble_acc:.4f}")
print(f"Best Single Model Test Accuracy: {test_acc:.4f}")
print(f"Improvement: +{ensemble_acc - test_acc:.4f}")

if ensemble_acc >= 0.93:
    print(f"\n{'='*60}")
    print(f"*** TARGET 93% MET! Ensemble: {ensemble_acc:.4f} ***")
    print(f"{'='*60}")
else:
    print(f"\nTarget: 0.93 | Ensemble Gap: {max(0, 0.93 - ensemble_acc):.4f}")

# Ensemble confusion matrix
cm_ens = confusion_matrix(labels[:len(ensemble_preds)], ensemble_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_ens, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Run 5 Confusion Matrix (Ensemble)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_ensemble_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Save best model and top ensemble models
save_ckpt(best5_state, f"{RUN5_DIR}/best_model_run5.pkl")
for i, (metrics, state) in enumerate(top3):
    save_ckpt(state, f"{RUN5_DIR}/ensemble_model_{i+1}_{metrics['best']:.4f}.pkl")

# Build results summary
run_results_5 = [f"Run {m['run']}: Best={m['best']:.4f}  Final={m['final_val']:.4f}"
                 for m in all5_metrics]

final_acc = max(test_acc, ensemble_acc)
status = 'TARGET MET!' if final_acc >= 0.93 else 'Below Target'

run5_metrics = "\n".join([
    f"### Run 5 (JAX/Flax VGG - Target 93%)",
    f"**Date**: {RUN5_TIMESTAMP}",
    f"**Status**: {status}",
    "",
    "### Configuration",
    "- Framework: JAX + Flax",
    "- Architecture: VGG-style (4 conv blocks: 64/128/256/512, GAP head)",
    "- Optimizer: Adam + CosineDecay (1e-3 -> 1e-6) + gradient clipping 1.0",
    "- Loss: SoftmaxCrossEntropy + LabelSmoothing(0.1)",
    "- Augmentation: Flip, Rotation(0.2), Zoom(0.15), Contrast(0.1), Translation(0.1)",
    "- Dropout: 0.2/0.25/0.3/0.4 (conv blocks), 0.5/0.3 (dense)",
    f"- Epochs per run: {EPOCHS_5}",
    f"- Total runs: {run_num}",
    f"- Batch: {BATCH_SIZE_PER_DEVICE}/device x {N_DEVICES} devices = {EFFECTIVE_BATCH}",
    "",
    "### Results",
    chr(10).join(run_results_5),
    "",
    "### Best Result",
    f"- Best Single Test Accuracy: {test_acc:.4f} (Run {best5_run})",
    f"- Ensemble Test Accuracy (Top {len(top3)}): {ensemble_acc:.4f}",
    f"- Target: 0.93 | Gap: {max(0, 0.93 - final_acc):.4f}",
    "---",
    "",
])

with open("RUN_TRACKING.md", "a") as f:
    f.write(run5_metrics)

print(f"\nRun 5 complete.")
print(f"Best single: {test_acc:.4f} | Ensemble: {ensemble_acc:.4f}")
print(f"Models saved to {RUN5_DIR}/")
print(f"RUN_TRACKING.md updated.")
